In [ ]:
!pip install flash-attn --no-build-isolation -q
!pip install pathway sentence-transformers transformers torch -q

In [2]:
# ============================================================================
# NOTEBOOK 3: PATHWAY RATIONALE GENERATOR - FINAL
# Using SAME syntax and patterns from Notebook 2 (WORKING)
# ============================================================================

import pathway as pw
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import json
from datetime import datetime
from tqdm.auto import tqdm
import gc
import os

In [3]:
EMBEDDING_CONFIG = {
    'model_name': 'Alibaba-NLP/gte-Qwen2-7B-instruct',
    'dimension': 3584,
    'batch_size': 4,
    'trust_remote_code': True,
    'normalize': True,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

DATASET_NAME = '/kaggle/input/kdsh26-pathway-vector-embeddings-books-qwen'
DATASET_BASE = f'{DATASET_NAME}/kaggle/working/pathway_storage'
METADATA_PATH = f"{DATASET_BASE}/vector_store_metadata.json"

ENSEMBLE_CSV = '/kaggle/input/submission-ensemble-qwen-deberta-inference/Submission_Ensemble_Qwen_Deberta_Inference.csv'
TEST_CSV = '/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/test.csv'
OUTPUT_CSV = '/kaggle/working/predictions_with_rationales.csv'
SUBMISSION_CSV = '/kaggle/working/submission.csv'

device = EMBEDDING_CONFIG['device']

# Load and fix metadata
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

# Fix paths
for book_name in metadata['books'].keys():
    metadata['books'][book_name]['embeddings_npy'] = metadata['books'][book_name]['embeddings_npy'].replace(
        '/kaggle/working/pathway_storage', DATASET_BASE
    )
    metadata['books'][book_name]['chunks_csv'] = metadata['books'][book_name]['chunks_csv'].replace(
        '/kaggle/working/pathway_storage', DATASET_BASE
    )

print("✅ Configuration and metadata loaded")


✅ Configuration and metadata loaded


In [4]:
print("\n" + "=" * 100)
print("🤖 LOADING QWEN MODEL FOR RATIONALE GENERATION")
print("=" * 100)

# Load Qwen model (using smaller variant for Kaggle)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # or "Qwen/Qwen2.5-3B-Instruct" for faster

print(f"\nLoading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✅ Qwen model loaded!")


🤖 LOADING QWEN MODEL FOR RATIONALE GENERATION

Loading Qwen/Qwen2.5-7B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Qwen model loaded!


In [5]:
# ============================================================================
# STEP 1: Reload Correct Ensemble Predictions
# ============================================================================

print("\n" + "=" * 100)
print("🔄 LOADING CORRECT ENSEMBLE PREDICTIONS")
print("=" * 100)

# Load the CORRECT ensemble file
ENSEMBLE_CSV = '/kaggle/input/submission-ensemble-qwen-deberta-inference/Submission_Ensemble_Qwen_Deberta_Inference.csv'
ensemble_df = pd.read_csv(ENSEMBLE_CSV)

print(f"✅ Loaded ensemble predictions: {len(ensemble_df)} rows")
print(f"   Columns: {list(ensemble_df.columns)}")

# Check label distribution
print(f"\n📊 Label Distribution:")
print(ensemble_df['label'].value_counts())

# Convert text labels to numeric if needed
if ensemble_df['label'].dtype == 'object':
    label_map = {'consistent': 0, 'contradict': 1}
    ensemble_df['label_numeric'] = ensemble_df['label'].map(label_map)
    print(f"\n✅ Converted text labels to numeric")
else:
    ensemble_df['label_numeric'] = ensemble_df['label']

# Load test data
TEST_CSV = '/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/test.csv'
test_df = pd.read_csv(TEST_CSV)
print(f"\n✅ Loaded test data: {len(test_df)} rows")

# Fix column name
if 'book_name' in test_df.columns:
    test_df = test_df.rename(columns={'book_name': 'bookname'})
    print(f"   ✅ Renamed 'book_name' → 'bookname'")

# Merge properly
merged_df = test_df.merge(ensemble_df, on='id', how='inner')
print(f"\n✅ Merged: {len(merged_df)} rows")
print(f"   Columns: {list(merged_df.columns)}")

# Show sample to verify
print(f"\n📋 Sample Merged Data:")
sample_cols = ['id', 'bookname', 'char', 'label']
print(merged_df[sample_cols].head(5))



🔄 LOADING CORRECT ENSEMBLE PREDICTIONS
✅ Loaded ensemble predictions: 60 rows
   Columns: ['id', 'label']

📊 Label Distribution:
label
consistent    50
contradict    10
Name: count, dtype: int64

✅ Converted text labels to numeric

✅ Loaded test data: 60 rows
   ✅ Renamed 'book_name' → 'bookname'

✅ Merged: 60 rows
   Columns: ['id', 'bookname', 'char', 'caption', 'content', 'label', 'label_numeric']

📋 Sample Merged Data:
    id                    bookname      char       label
0   95   The Count of Monte Cristo  Noirtier  contradict
1  136   The Count of Monte Cristo     Faria  consistent
2   59  In Search of the Castaways  Thalcave  consistent
3   60  In Search of the Castaways  Thalcave  consistent
4  124   The Count of Monte Cristo     Faria  contradict


In [6]:
# ============================================================================
# STEP 2: Load Metadata - SAME AS NOTEBOOK 2
# ============================================================================

print(f"\n📁 Loading metadata from: {METADATA_PATH}")

with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

print(f"✅ Metadata loaded")
print(f"   Books: {list(metadata['books'].keys())}")

# Fix paths - SAME AS NOTEBOOK 2
print("\n🔧 Fixing paths...")

for book_name in metadata['books'].keys():
    metadata['books'][book_name]['embeddings_npy'] = metadata['books'][book_name]['embeddings_npy'].replace(
        '/kaggle/working/pathway_storage',
        DATASET_BASE
    )
    metadata['books'][book_name]['chunks_csv'] = metadata['books'][book_name]['chunks_csv'].replace(
        '/kaggle/working/pathway_storage',
        DATASET_BASE
    )

if 'pathway_tables' in metadata:
    for book_name in metadata['pathway_tables'].keys():
        metadata['pathway_tables'][book_name]['parquet_path'] = metadata['pathway_tables'][book_name]['parquet_path'].replace(
            '/kaggle/working/pathway_storage',
            DATASET_BASE
        )

print("✅ Paths fixed!")



📁 Loading metadata from: /kaggle/input/kdsh26-pathway-vector-embeddings-books-qwen/kaggle/working/pathway_storage/vector_store_metadata.json
✅ Metadata loaded
   Books: ['In-search-of-the-castaways', 'The-count-of-monte-cristo']

🔧 Fixing paths...
✅ Paths fixed!


In [7]:
# ============================================================================
# STEP 2: Enhanced Rationale Generator with Qwen
# ============================================================================

class QwenRationaleGenerator:
    """Generate high-quality rationales using Qwen LLM + book evidence."""
    
    def __init__(self, metadata, qwen_model, tokenizer, embedding_config):
        self.metadata = metadata
        self.device = device
        self.embedding_config = embedding_config
        self.embeddings = {}
        self.chunks_df = {}
        self.qwen_model = qwen_model
        self.tokenizer = tokenizer
        
        print("Loading vector stores...")
        self._load_stores()
        
        print("\nLoading embedding model...")
        self.embedding_model = SentenceTransformer(
            self.embedding_config["model_name"],
            device=self.device,
            trust_remote_code=True,
            config_kwargs={"use_cache": False}
        )
        print("✅ Setup complete!")
    
    def _load_stores(self):
        """Load vector stores."""
        for book_name, book_meta in self.metadata['books'].items():
            emb_path = book_meta['embeddings_npy']
            chunks_path = book_meta['chunks_csv']
            
            embeddings = np.load(emb_path)
            chunks = pd.read_csv(chunks_path)
            
            self.embeddings[book_name] = embeddings
            self.chunks_df[book_name] = chunks
            
            print(f"  ✓ {book_name}: {len(chunks)} chunks")
    
    def _retrieve_context(self, query: str, book_name: str, top_k: int = 5):
        """Retrieve relevant passages from book."""
        normalized_book = normalize_book_name(book_name)
        
        if not query or not normalized_book or normalized_book not in self.embeddings:
            return []
        
        try:
            query_emb = self.embedding_model.encode([query], show_progress_bar=False)[0]
            book_embs = self.embeddings[normalized_book]
            book_chunks = self.chunks_df[normalized_book]
            
            similarities = np.dot(book_embs, query_emb)
            top_indices = np.argsort(similarities)[-top_k:][::-1]
            
            results = []
            for idx in top_indices:
                results.append({
                    'chunk': book_chunks.iloc[idx]['chunk'],
                    'score': float(similarities[idx]),
                    'book': normalized_book
                })
            
            return results
        except Exception as e:
            return []
    
    def generate_rationale_with_qwen(self, test_row, prediction_label):
        """Generate explanatory rationale using Qwen LLM."""
        char = test_row.get('char', 'Unknown')
        book_name = test_row.get('bookname', 'Unknown')
        content = test_row.get('content', '')
        caption = test_row.get('caption', '')
        
        # Retrieve evidence
        query = f"Character: {char}. Context: {caption}. Statement: {content}"
        passages = self._retrieve_context(query, book_name, top_k=5)
        
        # Prepare evidence text
        if passages and len(passages) > 0:
            evidence_text = "\n\n".join([
                f"Passage {i+1} (relevance: {p['score']:.2f}):\n{p['chunk'][:400]}"
                for i, p in enumerate(passages[:3])
            ])
        else:
            evidence_text = "No direct passages retrieved from the book."
        
        # Create prompt for Qwen
        prompt = f"""You are a literary analyst. Analyze whether a statement about a book character is consistent or contradicts the book.

Book: {book_name}
Character: {char}
Context: {caption}

Statement to verify:
"{content}"

Evidence from the book:
{evidence_text}

Classification: This statement **{prediction_label}s** the book.

Task: Write a clear, logical rationale (2-3 sentences) explaining WHY this statement {prediction_label}s the book based on the evidence. Be specific about what in the book supports or refutes the claim.

Rationale:"""

        # Generate with Qwen
        messages = [
            {"role": "system", "content": "You are a precise literary analyst who provides clear, evidence-based explanations."},
            {"role": "user", "content": prompt}
        ]
        
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.device)
        
        with torch.no_grad():
            generated_ids = self.qwen_model.generate(
                **model_inputs,
                max_new_tokens=200,
                temperature=0.7,
                top_p=0.9,
                do_sample=True
            )
        
        generated_ids = [
            output_ids[len(input_ids):] 
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        rationale = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Clean up
        rationale = rationale.strip()
        
        # Ensure it starts properly
        if not rationale.startswith("The statement"):
            if prediction_label == "contradict":
                rationale = f"The statement contradicts the novel. {rationale}"
            else:
                rationale = f"The statement is consistent with the novel. {rationale}"
        
        return rationale

print("✅ QwenRationaleGenerator class defined")


✅ QwenRationaleGenerator class defined


In [8]:
# ============================================================================
# STEP 4: Initialize Generator
# ============================================================================

print("\n" + "=" * 100)
print("🔧 INITIALIZING QWEN RATIONALE GENERATOR")
print("=" * 100)

generator_qwen = QwenRationaleGenerator(
    metadata=metadata,
    qwen_model=qwen_model,
    tokenizer=tokenizer,
    embedding_config=EMBEDDING_CONFIG
)

print("\n✅ Generator ready!")


🔧 INITIALIZING QWEN RATIONALE GENERATOR
Loading vector stores...
  ✓ In-search-of-the-castaways: 375 chunks
  ✓ The-count-of-monte-cristo: 1290 chunks

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/284 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/370 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

✅ Setup complete!

✅ Generator ready!


In [10]:
def normalize_book_name(book_name):
    """
    Normalize book names to match embedding keys.
    'The Count of Monte Cristo' -> 'The-count-of-monte-cristo'
    """
    if pd.isna(book_name) or not book_name:
        return None
    
    # Replace spaces with hyphens
    normalized = book_name.strip().replace(' ', '-')
    
    # Split and handle capitalization
    parts = normalized.split('-')
    
    if parts:
        # Capitalize only first letter of first word
        parts[0] = parts[0][0].upper() + parts[0][1:].lower() if len(parts[0]) > 0 else parts[0]
        # Rest all lowercase
        parts[1:] = [p.lower() for p in parts[1:]]
    
    return '-'.join(parts)

print("✅ Normalization function defined")

# Test it
print("\nTesting normalization:")
for name in ["The Count of Monte Cristo", "In Search of the Castaways"]:
    print(f"  '{name}' -> '{normalize_book_name(name)}'")


✅ Normalization function defined

Testing normalization:
  'The Count of Monte Cristo' -> 'The-count-of-monte-cristo'
  'In Search of the Castaways' -> 'In-search-of-the-castaways'


In [11]:
# ============================================================================
# STEP 5: Generate Enhanced Rationales
# ============================================================================

from tqdm.auto import tqdm

print("\n" + "=" * 100)
print("🎯 GENERATING QWEN-ENHANCED RATIONALES")
print("=" * 100)

print(f"\nProcessing {len(merged_df)} predictions...")
print("(This may take 10-15 minutes for better quality)\n")

rationales_qwen = []

for idx, row in tqdm(merged_df.iterrows(), total=len(merged_df), desc="Generating"):
    # Get label
    label_val = row['label']
    if isinstance(label_val, str):
        prediction_label = label_val.lower()
    else:
        prediction_label = "contradict" if label_val == 1 else "consistent"
    
    # Generate rationale
    rationale = generator_qwen.generate_rationale_with_qwen(row, prediction_label)
    rationales_qwen.append(rationale)

merged_df['rationale'] = rationales_qwen

print("\n✅ All rationales generated!")

# Show samples
print(f"\n{'='*100}")
print("📋 QWEN-ENHANCED RATIONALE SAMPLES")
print('='*100)

for i in range(min(3, len(merged_df))):
    print(f"\n{'-'*100}")
    print(f"ID: {merged_df.iloc[i]['id']} | Prediction: {merged_df.iloc[i]['label']}")
    print(f"Book: {merged_df.iloc[i]['bookname']} | Char: {merged_df.iloc[i]['char']}")
    print(f"\nRationale:")
    print(merged_df.iloc[i]['rationale'])



🎯 GENERATING QWEN-ENHANCED RATIONALES

Processing 60 predictions...
(This may take 10-15 minutes for better quality)



Generating:   0%|          | 0/60 [00:00<?, ?it/s]


✅ All rationales generated!

📋 QWEN-ENHANCED RATIONALE SAMPLES

----------------------------------------------------------------------------------------------------
ID: 95 | Prediction: contradict
Book: The Count of Monte Cristo | Char: Noirtier

Rationale:
The statement contradicts the novel. This statement contradicts the book because there is no evidence that Noirtier pre-emptively handed the conspiracy dossier to a British spy. The passages provided do not mention any such action by Noirtier. Furthermore, Noirtier is portrayed as a calm and composed figure, as evidenced by his behavior during the conversation with Villefort and his departure from the room with "the same calmness that had characterized him during the whole of this remarkable and trying conversation." These details suggest that Noirtier did not engage in any clandestine activities to hand over sensitive information to a foreign agent.

---------------------------------------------------------------------------------

In [12]:
# ============================================================================
# STEP 6: Create Final Submission
# ============================================================================

submission = pd.DataFrame({
    'Story ID': merged_df['id'],
    'Prediction': merged_df['label'],
    'Rationale': merged_df['rationale']
})

FINAL_SUBMISSION = '/kaggle/working/submission_qwen_enhanced.csv'
submission.to_csv(FINAL_SUBMISSION, index=False)

print(f"\n{'='*100}")
print("✅ SUBMISSION CREATED")
print('='*100)
print(f"File: {FINAL_SUBMISSION}")
print(f"Rows: {len(submission)}")
print(f"\n✅ Ready to submit!")


✅ SUBMISSION CREATED
File: /kaggle/working/submission_qwen_enhanced.csv
Rows: 60

✅ Ready to submit!


In [ ]:
# ============================================================================
# STEP 11: Cleanup GPU Memory
# ============================================================================

print("\n" + "=" * 100)
print("🧹 CLEANUP")
print("=" * 100)

print("\n🧹 Cleaning up GPU memory...")

# Delete models
if hasattr(generator, 'embedding_model'):
    del generator.embedding_model
if hasattr(generator, 'generator'):
    del generator.generator

gc.collect()
torch.cuda.empty_cache()

final_vram_allocated = torch.cuda.memory_allocated() / 1024**3
final_vram_reserved = torch.cuda.memory_reserved() / 1024**3

print(f"✅ GPU memory cleaned")
print(f"   Allocated: {final_vram_allocated:.2f} GB")
print(f"   Reserved: {final_vram_reserved:.2f} GB")
